## Transformer Fine-Tuning for Sentiment Classification (IMDb)

<center>

**Name:** Marrion Kiprop Cherop  
**Reg No:** ST62/80971/2024  
**Programme:** MSc Artificial Intelligence  
**Course:** CSA 803 Natural Language Processing  
**Module:** Module 8: Transformer Models and Pretrained Models  

<center>


### Mini-project Assignment
Fine-tuning a pretrained transformer (DistilBERT) on the IMDb 50,000-review sentiment dataset, and evaluate it against accuracy, precision, recall, and F1.

### Process
1. Loading the IMDb dataset from Hugging Face.
2. Loading a pretrained DistilBERT tokenizer and model.
3. Tokenize with padding and truncation.
4. Fine-tune using the Hugging Face `Trainer` API.
5. Evaluate on accuracy, precision, recall, and F1, and interrogate what the numbers actually mean.

### A note on the dataset loader
This assignment uses `stanfordnlp/imdb` to load the dataset


## Environment setup
Install the libraries the task specifies.

In [ ]:
!pip install -q transformers datasets torch scikit-learn accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {device}")
if device == "cpu":
    print("No GPU detected. Keep SUBSET_SIZE small below, or switch to a GPU runtime before the full run.")


Device in use: cpu
No GPU detected. Keep SUBSET_SIZE small below, or switch to a GPU runtime before the full run.


## 1. Load the dataset

`stanfordnlp/imdb` gives 25,000 train and 25,000 test reviews, evenly split between positive and negative — no class imbalance problem to solve here, which simplifies the metric story later.

In [ ]:
try:
    raw_datasets = load_dataset("stanfordnlp/imdb")
except Exception as e:
    print(f"stanfordnlp/imdb failed ({e}); falling back to legacy 'imdb' identifier.")
    raw_datasets = load_dataset("imdb")

# Drop the unsupervised split if present -- not needed for supervised fine-tuning
raw_datasets.pop("unsupervised", None)

print(raw_datasets)
print("\nLabel balance (train):")
train_labels = raw_datasets["train"]["label"]
print(f"  positive: {sum(train_labels)}  negative: {len(train_labels) - sum(train_labels)}")
print("\nSample review (truncated):")
print(raw_datasets["train"][0]["text"][:300], "...")
print("Label:", raw_datasets["train"][0]["label"], "(0=neg, 1=pos)")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})

Label balance (train):
  positive: 12500  negative: 12500

Sample review (truncated):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...
Label: 0 (0=neg, 1=pos)


## 2. Load a pretrained model and tokenizer

DistilBERT (`distilbert-base-uncased`) rather than full BERT: ~40% fewer parameters, ~60% faster inference, and it retains about 97% of BERT's language-understanding performance on GLUE-style benchmarks per the original DistilBERT paper. For a binary sentiment task on review text, that trade is worth it — the accuracy ceiling here is set by the task and data, not by model depth. `num_labels=2` tells the classification head to produce two logits (negative/positive).

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print(f"Loaded {MODEL_NAME} with {model.num_parameters():,} parameters.")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded distilbert-base-uncased with 66,955,010 parameters.


## 3. Preprocess and tokenize

Padding and truncation are both necessary and for different reasons. Truncation (`max_length=256`) caps review length — IMDb reviews run long, some past 1,000 tokens, and DistilBERT's positional embeddings only go to 512; 256 keeps most of the signal (sentiment is usually established early and reinforced throughout) while keeping training time manageable. Padding (`padding="max_length"`) makes every sequence in a batch the same length, which is what lets the model process a batch as a single tensor rather than one example at a time. The attention mask that comes out of the tokenizer tells the model which tokens are real content and which are padding, so padding doesn't leak into the learned representation.

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print(tokenized_datasets)
print("\nSample tokenized input_ids (first 20 tokens):")
print(tokenized_datasets["train"][0]["input_ids"][:20])


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})

Sample tokenized input_ids (first 20 tokens):
tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009])


## 3a. Subsample for a feasibility run


In [ ]:
SUBSET_SIZE = None  # set to None for the full 25,000/25,000 but split if you want a smaller sample

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"]

if SUBSET_SIZE is not None:
    train_dataset = train_dataset.shuffle(seed=42).select(range(SUBSET_SIZE))
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(min(SUBSET_SIZE, len(eval_dataset))))

print(f"Train examples: {len(train_dataset)}  |  Eval examples: {len(eval_dataset)}")


Train examples: 25000  |  Eval examples: 25000


## 4. Fine-tune with the `Trainer` API

`compute_metrics` runs at every evaluation step and is where accuracy, precision, recall, and F1 get computed — `average="binary"` is correct here because this is a two-class problem and both classes are equally represented, so there's no need for macro/weighted averaging to correct for imbalance.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    bf16=False, # Explicitly disable bfloat16 to avoid XLA conflicts
    optim="adamw_torch", # Use 'adamw_torch' as 'adamw_hf' is not a valid option
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.246359,0.237379,0.906120,0.931638,0.876560,0.903260
2,0.097737,0.312889,0.914320,0.907603,0.922560,0.915020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.20751274249832827, metrics={'train_runtime': 529.4018, 'train_samples_per_second': 94.476, 'train_steps_per_second': 5.905, 'total_flos': 3311684966400000.0, 'train_loss': 0.20751274249832827, 'epoch': 2.0})


## 5. Evaluate

Beyond the four headline metrics, the confusion matrix and per-class report matter because accuracy alone can hide an asymmetric failure mode — a model that's confidently right on obvious reviews but wrong on sarcastic or mixed-sentiment ones will still show a respectable overall accuracy. Look at where the errors actually land before trusting the top-line number.

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation metrics:")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

predictions_output = trainer.predict(eval_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))

print("Confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_true, y_pred))


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.097737,0.312889,2,0.914320,0.907603,0.922560,0.915020


Evaluation metrics:
  eval_loss: 0.3129
  eval_accuracy: 0.9143
  eval_precision: 0.9076
  eval_recall: 0.9226
  eval_f1: 0.9150


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification report:
              precision    recall  f1-score   support

    negative       0.92      0.91      0.91     12500
    positive       0.91      0.92      0.92     12500

    accuracy                           0.91     25000
   macro avg       0.91      0.91      0.91     25000
weighted avg       0.91      0.91      0.91     25000

Confusion matrix (rows = actual, cols = predicted):
[[11326  1174]
 [  968 11532]]


The full run on 25,000/25,000 examples delivers 91.43% accuracy, 90.76% precision, 92.26% recall, and 91.50% F1, and every one of those numbers moves up from the 2,000-example subset (88.00%, 87.70%, 88.40%, 88.05%), which confirms the extra data earns its keep rather than just adding runtime. The overfitting signature from the subset run persists at full scale — training loss drops from 0.2464 to 0.0977 across the two epochs while validation loss climbs from 0.2374 to 0.3129 — which settles the question of whether that divergence was a small-data artifact: it wasn't, DistilBERT hits its useful capacity for this task within two epochs regardless of data volume, and `metric_for_best_model="f1"` correctly retains epoch 2's checkpoint anyway because F1 keeps rising (0.9033 → 0.9150) even as loss worsens, proving loss and target metric are not tracking the same signal here. The confusion matrix — [[11326, 1174], [968, 11532]] — carries a mild but real directional bias, 1,174 negative reviews called positive against 968 positive reviews called negative, which is exactly what drags precision (0.9076) below recall (0.9226) on an otherwise perfectly balanced test set, and the next required step is pulling actual false positives to confirm whether backhanded reviews (praise for an actor inside a negative verdict on the film) are driving it, not asserting that from the numbers alone. Training cost stands at 529.4 seconds for 3,126 steps at 94.5 samples/second, nearly triple the subset's 33.8 samples/second, a gain that comes from XLA's compilation overhead being amortized across twelve times more steps rather than from any change in batch size, and it establishes TPU efficiency as a function of run length as much as data volume. None of this licenses a direct comparison against the earlier VADER (~71%) and TF-IDF/Logistic Regression (~75%) results from the CSA 803 product-review notebook, because that was a three-class problem on a different domain and dataset — the honest claim is that IMDb binary sentiment is a structurally easier separation task and that contextual fine-tuning adds real headroom within this task, not that DistilBERT outperformed the earlier methods by twenty points on equivalent ground.